# Fiber Analysis Pipeline

ND2 file → fiber segmentation → CSV + intensity profiles (NPZ)

Run cells top to bottom. Each step caches results so you can re-run without starting over.

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install tifffile nd2 scipy scikit-image joblib pandas micro_sam
!pip install git+https://github.com/MIC-DKFZ/MedNeXt.git

In [ ]:
import os
if not os.path.exists("pytorch_connectomics"):
    !git lfs install
    !git clone https://github.com/zhangdjr/pytorch_connectomics.git
os.chdir("pytorch_connectomics")
!pip install -e .

## Configuration — edit the ND2 path below

In [ ]:
import os, sys, glob, subprocess, csv as csv_module
from pathlib import Path
import numpy as np
import tifffile

# >>> EDIT THIS: path to your ND2 file <<<
ND2_FILE = "/path/to/your/sample.nd2"

REPO_DIR = Path(".").resolve()
CHECKPOINT = REPO_DIR / "outputs/fiber_retrain_all/20260311_223801/checkpoints/last.ckpt"
OUTPUT_BASE = REPO_DIR / "fiber_results"

ND2_BASENAME = Path(ND2_FILE).stem
OUTPUT_DIR = OUTPUT_BASE / ND2_BASENAME
TILE_DIR = OUTPUT_DIR / "tiles"
FIBER_SEG_DIR = OUTPUT_DIR / "fiber_seg"
CACHE_DIR = OUTPUT_DIR / "cache"

for d in [TILE_DIR, FIBER_SEG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for p in [str(REPO_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

assert Path(ND2_FILE).exists(), f"ND2 file not found: {ND2_FILE}"
assert CHECKPOINT.exists(), f"Model checkpoint not found: {CHECKPOINT} (did git lfs pull run?)"

## Step 1: Extract tiles from ND2

In [ ]:
from extract_nd2_tile import extract_all_channels
tile_names = extract_all_channels(ND2_FILE, str(TILE_DIR))
print(f"{len(tile_names)} tiles: {tile_names}")

## Step 2: Fiber segmentation inference (GPU, ~10 min/tile)

In [ ]:
existing = {Path(f).stem.replace("_ch1_prediction", "") for f in glob.glob(str(FIBER_SEG_DIR / "*_prediction.tiff"))}
to_infer = [t for t in tile_names if t not in existing]

if not to_infer:
    print("All tiles already have predictions, skipping.")
else:
    img_lines = "".join(f"    - {TILE_DIR}/{t}_ch1.tif
" for t in to_infer)
    base_yaml = REPO_DIR / "tutorials" / "bases" / "mednext.yaml"

    yaml_content = f"""_base_: {base_yaml}
experiment_name: fiber_retrain_all
system:
  inference: {{num_cpus: 4, num_workers: 4, batch_size: 1}}
  seed: 42
model:
  out_channels: 3
  input_size: [32, 96, 96]
  output_size: [32, 96, 96]
  mednext_size: S
  mednext_kernel_size: 3
  deep_supervision: false
  loss_functions: [WeightedBCEWithLogitsLoss, DiceLoss, WeightedBCEWithLogitsLoss, DiceLoss, WeightedMSELoss]
  loss_weights: [4.0, 2.0, 1.0, 0.5, 2.0]
  loss_kwargs:
  - {{reduction: mean, pos_weight: 20.0}}
  - {{sigmoid: true, smooth_nr: 1e-5, smooth_dr: 1e-5}}
  - {{reduction: mean}}
  - {{sigmoid: true, smooth_nr: 1e-5, smooth_dr: 1e-5}}
  - {{tanh: true}}
  multi_task_config:
  - [0, 1, binary, [0, 1]]
  - [1, 2, instance_boundary, [2, 3]]
  - [2, 3, skeleton_aware_edt, [4]]
data:
  image_transform: {{clip_percentile_low: 0.005, clip_percentile_high: 0.995}}
test:
  data:
    test_image:
{img_lines}    output_path: "{FIBER_SEG_DIR}"
    image_transform: {{normalize: "0-1", clip_percentile_low: 0.005, clip_percentile_high: 0.995}}
inference:
  sliding_window: {{window_size: [32,256,256], stride: [16,128,128], blending: gaussian, sigma_scale: 0.25, padding_mode: reflect, pad_size: [16,32,32]}}
  test_time_augmentation:
    enabled: true
    channel_activations: [[0,1,sigmoid],[1,2,sigmoid],[2,3,tanh]]
  decoding:
  - name: decode_instance_binary_contour_distance
    kwargs:
      binary_threshold: [0.7503891044149624, 0.0046306263327246]
      contour_threshold: [0.48708101851244906, 1.025119772551816]
      distance_threshold: [-0.6695257662389609, -0.07270575159140885]
      min_instance_size: 100
      min_seed_size: 40
  save_prediction: {{output_formats: [tiff]}}
  evaluation: {{enabled: false}}
"""
    config_path = OUTPUT_DIR / "inference_config.yaml"
    config_path.write_text(yaml_content)

    result = subprocess.run(
        [sys.executable, "-u", str(REPO_DIR / "scripts" / "main.py"),
         "--config", str(config_path), "--mode", "test", "--checkpoint", str(CHECKPOINT)],
        cwd=str(REPO_DIR))
    assert result.returncode == 0, "Inference failed!"

    for f in glob.glob(str(FIBER_SEG_DIR / "*_tta_prediction.*")):
        os.remove(f)

## Step 3: Cell body segmentation (micro-sam)

In [ ]:
try:
    from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation
    HAS_MICROSAM = True
except ImportError:
    HAS_MICROSAM = False
    print("micro_sam not installed, skipping cell segmentation")

for tile in tile_names:
    cache_path = CACHE_DIR / f"{tile}_cell_seg.npz"
    if cache_path.exists():
        continue
    dapi = tifffile.imread(str(TILE_DIR / f"{tile}_ch0_dapi.tif"))
    if not HAS_MICROSAM:
        np.savez_compressed(str(cache_path), cell_seg=np.zeros(dapi.shape, dtype=np.int32))
        continue
    predictor, segmenter = get_predictor_and_segmenter("vit_b_lm")
    seg = automatic_instance_segmentation(predictor, segmenter, input_path=dapi, verbose=True).astype(np.int32)
    np.savez_compressed(str(cache_path), cell_seg=seg)

## Step 4: Fiber analysis

In [ ]:
from fiber_pipeline import get_config, run_pipeline

config = get_config()
config["tile_dir"] = str(TILE_DIR)
config["pred_dir"] = str(FIBER_SEG_DIR)
config["output_dir"] = str(OUTPUT_BASE)
config["n_jobs"] = 16

for tile in tile_names:
    run_pipeline(tile, ND2_BASENAME, config)

## Step 5: Combine results

In [ ]:
combined_csv = OUTPUT_DIR / f"{ND2_BASENAME}_combined.csv"
all_rows, header = [], None
for csv_file in sorted(OUTPUT_DIR.glob(f"{ND2_BASENAME}_*.csv")):
    if "combined" in csv_file.name:
        continue
    with open(csv_file) as f:
        reader = csv_module.DictReader(f)
        if header is None:
            header = reader.fieldnames
        all_rows.extend(list(reader))
with open(combined_csv, "w", newline="") as f:
    w = csv_module.DictWriter(f, fieldnames=header)
    w.writeheader()
    w.writerows(all_rows)

all_fids, all_valid, all_tiles_arr, ch_all = [], [], [], {}
for pf in sorted(OUTPUT_DIR.glob(f"{ND2_BASENAME}_*_profiles.npz")):
    d = np.load(str(pf))
    t = pf.stem.replace(f"{ND2_BASENAME}_", "").replace("_profiles", "")
    all_fids.append(d["fiber_ids"])
    all_valid.append(d["is_valid"])
    all_tiles_arr.extend([t] * len(d["fiber_ids"]))
    for k in d.files:
        if k not in ("fiber_ids", "is_valid"):
            ch_all.setdefault(k, []).append(d[k])
combined_npz = OUTPUT_DIR / f"{ND2_BASENAME}_combined_profiles.npz"
np.savez_compressed(str(combined_npz),
    fiber_ids=np.concatenate(all_fids), is_valid=np.concatenate(all_valid),
    tile_names=np.array(all_tiles_arr), **{k: np.concatenate(v) for k, v in ch_all.items()})

print(f"CSV: {combined_csv} ({len(all_rows)} fibers)")
print(f"Profiles: {combined_npz}")

# Clean up tiles to save disk space (can always re-extract from ND2)
import shutil
if TILE_DIR.exists():
    shutil.rmtree(TILE_DIR)
    print(f"Deleted tiles/ to save space")